# 🛡️ SentinelID — Master A100 Training Notebook

Runs all 7 modules back-to-back. Checkpoints persist to Google Drive.

**Module order:**
- M1: Passive 3D Liveness (DepthLivenessModel / ResNet-50)
- M2: Deepfake Detection (EfficientNet-B4 + FFT)
- M3: Face Recognition (ArcFace / iResNet-100)
- M4: Behavioral Analysis (AU-GNN + Gaze)
- M5: Document Intelligence (LayoutLM-style)
- M6: Score Fusion (Calibrated MLP)
- M7: Edge Distillation (MobileNetV3 → ONNX)

**Estimated time on A100:** ~10–15 h for real datasets, ~45 min for synthetic.

## 0 · Setup — Run Once

In [ ]:
# ── Mount Drive ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_CKPT = pathlib.Path('/content/drive/MyDrive/sentinelid/checkpoints')
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Checkpoint dir:', DRIVE_CKPT)

In [ ]:
# ── Clone / pull repo ─────────────────────────────────────────────────────────
import subprocess, os

REPO = '/content/SentinelID'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Aprameya05/SentinelID.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase=false'], check=True)

os.chdir(REPO)
print('Repo at:', os.getcwd())

In [ ]:
# ── Install deps ──────────────────────────────────────────────────────────────
!pip install -q timm omegaconf wandb rich einops onnx onnxruntime torchmetrics
!pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__.split("+")[0])')+cu121.html
print('Dependencies installed.')

In [ ]:
# ── W&B login ─────────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste your API key when prompted

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1 · Synthetic Dataset Generator (use when real data unavailable)

In [ ]:
"""Generate synthetic liveness, deepfake, and behavioral datasets for pipeline testing."""
import numpy as np
from PIL import Image
from pathlib import Path

def make_synthetic_liveness(root='/content/data/liveness', n_per_class=500, img_size=224):
    """Live = natural face-like noise, spoof = flat gradient (simulates print)."""
    rng = np.random.default_rng(42)
    for split in ('train', 'val'):
        n = n_per_class if split == 'train' else n_per_class // 5
        for label in ('live', 'spoof'):
            d = Path(root) / split / label
            d.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                if label == 'live':
                    arr = (rng.normal(128, 40, (img_size, img_size, 3))
                           .clip(0, 255).astype(np.uint8))
                else:
                    base = rng.integers(50, 200, 3)
                    arr = np.tile(base, (img_size, img_size, 1)).astype(np.uint8)
                    arr += rng.integers(0, 10, arr.shape, dtype=np.uint8)
                Image.fromarray(arr).save(d / f'{label}_{i:05d}.jpg')
    total = sum(1 for _ in Path(root).rglob('*.jpg'))
    print(f'Synthetic liveness: {total} images in {root}')

def make_synthetic_deepfake(root='/content/data/deepfake', n_per_class=400, img_size=224):
    """Real = natural texture, fake = smooth blend (simulates GAN output)."""
    rng = np.random.default_rng(7)
    for split in ('train', 'val'):
        n = n_per_class if split == 'train' else n_per_class // 5
        for label in ('real', 'fake'):
            d = Path(root) / split / label
            d.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                if label == 'real':
                    arr = rng.integers(0, 256, (img_size, img_size, 3), dtype=np.uint8)
                else:
                    # Smooth (upscaled low-freq) — GAN-like
                    small = rng.integers(0, 256, (16, 16, 3), dtype=np.uint8)
                    arr = np.array(Image.fromarray(small).resize(
                        (img_size, img_size), Image.BILINEAR), dtype=np.uint8)
                Image.fromarray(arr).save(d / f'{label}_{i:05d}.jpg')
    print(f'Synthetic deepfake: done → {root}')

def make_synthetic_face(root='/content/data/faces', n_ids=200, n_per_id=10, img_size=112):
    """n_ids identities, n_per_id images each."""
    rng = np.random.default_rng(99)
    for split in ('train', 'val'):
        n_ids_split = n_ids if split == 'train' else max(20, n_ids // 5)
        for uid in range(n_ids_split):
            d = Path(root) / split / f'id_{uid:05d}'
            d.mkdir(parents=True, exist_ok=True)
            base_color = rng.integers(40, 220, 3)
            for j in range(n_per_id):
                arr = np.clip(
                    np.tile(base_color, (img_size, img_size, 1)) + rng.integers(-30, 30, (img_size, img_size, 3)),
                    0, 255
                ).astype(np.uint8)
                Image.fromarray(arr).save(d / f'{j:04d}.jpg')
    print(f'Synthetic face IDs: {n_ids} train, done → {root}')

make_synthetic_liveness()
make_synthetic_deepfake()
make_synthetic_face()
print('\nAll synthetic datasets ready.')

## M1 · Passive 3D Liveness

In [ ]:
# Write Colab-compatible liveness config
import yaml
cfg_liveness = {
    'project': {'name': 'sentinelid-liveness', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'resnet50'},
    'data': {
        'image_size': 224,
        'datasets': [
            {'name': 'liveness_train', 'root': '/content/data/liveness'}
        ]
    },
    'training': {
        'batch_size': 32, 'epochs': 30, 'lr': 1e-3,
        'weight_decay': 1e-4, 'bce_weight': 1.0,
        'depth_weight': 0.5, 'contrastive_weight': 0.1,
        'contrastive_margin': 1.0, 'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/liveness_config.yaml', 'w') as f:
    yaml.dump(cfg_liveness, f)
print('Liveness config written.')

In [ ]:
!cd /content/SentinelID && python training/train_liveness.py --config configs/liveness_config.yaml

## M2 · Deepfake Detection

In [ ]:
cfg_deepfake = {
    'project': {'name': 'sentinelid-deepfake', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'efficientnet_b4', 'pretrained': True},
    'data': {
        'image_size': 224,
        'datasets': [
            {'name': 'deepfake_train', 'root': '/content/data/deepfake'}
        ]
    },
    'training': {
        'batch_size': 32, 'epochs': 20, 'lr': 5e-4,
        'weight_decay': 1e-4, 'focal_gamma': 2.0,
        'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/deepfake_config.yaml', 'w') as f:
    yaml.dump(cfg_deepfake, f)
print('Deepfake config written.')

In [ ]:
!cd /content/SentinelID && python training/train_deepfake.py --config configs/deepfake_config.yaml

## M3 · ArcFace Face Recognition

In [ ]:
cfg_face = {
    'project': {'name': 'sentinelid-face', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'iresnet50', 'embedding_dim': 512,
              'num_classes': 200, 'pretrained': False},
    'data': {
        'image_size': 112,
        'datasets': [
            {'name': 'faces', 'root': '/content/data/faces'}
        ]
    },
    'training': {
        'batch_size': 64, 'epochs': 25, 'lr': 1e-3,
        'weight_decay': 5e-4, 's': 64.0, 'm': 0.5,
        'eval_every_n_epochs': 5
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/arcface_config.yaml', 'w') as f:
    yaml.dump(cfg_face, f)
print('ArcFace config written.')

In [ ]:
!cd /content/SentinelID && python training/train_face_recognition.py --config configs/arcface_config.yaml

## M4 · Behavioral Analysis (AU-GNN)

In [ ]:
# Generate synthetic landmark data for behavioral training
import numpy as np, torch
from pathlib import Path

beh_root = Path('/content/data/behavioral')
beh_root.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
n = 2000
# 68 facial landmarks, 2D coords normalized to [0,1]
landmarks = rng.uniform(0, 1, (n, 68, 2)).astype(np.float32)
# 12 AUs, intensity 0-5 (float)
au_labels = rng.uniform(0, 5, (n, 12)).astype(np.float32)
# Gaze: (yaw, pitch) in radians
gaze_labels = rng.uniform(-0.5, 0.5, (n, 2)).astype(np.float32)

np.save(beh_root / 'landmarks.npy', landmarks)
np.save(beh_root / 'au_labels.npy', au_labels)
np.save(beh_root / 'gaze_labels.npy', gaze_labels)
print(f'Behavioral data: {n} samples → {beh_root}')

In [ ]:
cfg_behavioral = {
    'project': {'name': 'sentinelid-behavioral', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'hidden_dim': 256, 'n_layers': 3, 'n_heads': 4},
    'data': {
        'root': '/content/data/behavioral',
        'val_split': 0.15
    },
    'training': {
        'batch_size': 128, 'epochs': 30, 'lr': 1e-3,
        'weight_decay': 1e-4, 'au_weight': 1.0, 'gaze_weight': 0.5
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/behavioral_config.yaml', 'w') as f:
    yaml.dump(cfg_behavioral, f)
!cd /content/SentinelID && python training/train_behavioral.py --config configs/behavioral_config.yaml

## M5 · Document Intelligence

In [ ]:
# Generate synthetic document images
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from pathlib import Path

doc_root = Path('/content/data/documents')
rng = np.random.default_rng(13)

for split in ('train', 'val'):
    n = 300 if split == 'train' else 60
    for label in ('genuine', 'forged'):
        d = doc_root / split / label
        d.mkdir(parents=True, exist_ok=True)
        for i in range(n):
            img = Image.new('RGB', (224, 224),
                           color=tuple(rng.integers(240, 255, 3).tolist()))
            draw = ImageDraw.Draw(img)
            # Draw text-like lines
            for row in range(5, 200, 20):
                w = rng.integers(80, 200)
                color = (0, 0, 0) if label == 'genuine' else tuple(rng.integers(100, 180, 3).tolist())
                draw.rectangle([10, row, 10+w, row+8], fill=color)
            if label == 'forged':
                # Add noise artifacts to simulate forgery
                arr = np.array(img)
                arr += rng.integers(0, 30, arr.shape, dtype=np.uint8)
                img = Image.fromarray(arr.clip(0, 255).astype(np.uint8))
            img.save(d / f'{label}_{i:04d}.jpg')

print(f'Document dataset: {sum(1 for _ in doc_root.rglob("*.jpg"))} images → {doc_root}')

In [ ]:
cfg_document = {
    'project': {'name': 'sentinelid-document', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'model': {'backbone': 'resnet50', 'num_classes': 2},
    'data': {
        'image_size': 224,
        'root': '/content/data/documents'
    },
    'training': {
        'batch_size': 32, 'epochs': 20, 'lr': 5e-4,
        'weight_decay': 1e-4
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/document_config.yaml', 'w') as f:
    yaml.dump(cfg_document, f)
!cd /content/SentinelID && python training/train_document.py --config configs/document_config.yaml

## M6 · Score Fusion

In [ ]:
# Generate synthetic score vectors for fusion training
import numpy as np
from pathlib import Path

score_root = Path('/content/data/score_vectors')
score_root.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
n = 5000
# 5 module scores: liveness, deepfake_real, face_match, au_signal, doc_score
labels = rng.integers(0, 2, n)  # 0=reject, 1=accept
scores = np.zeros((n, 5), dtype=np.float32)
for i, lbl in enumerate(labels):
    if lbl == 1:  # genuine
        scores[i] = rng.beta(8, 2, 5)  # skewed high
    else:         # impostor/attack
        scores[i] = rng.beta(2, 8, 5)  # skewed low

np.save(score_root / 'scores.npy', scores)
np.save(score_root / 'labels.npy', labels)
print(f'Fusion data: {n} samples (genuine={labels.sum()}, reject={n-labels.sum()})')

In [ ]:
cfg_fusion = {
    'project': {'name': 'sentinelid-fusion', 'device': 'cuda', 'mixed_precision': False},
    'paths': {
        'checkpoint_dir': str(DRIVE_CKPT),
        'score_cache_dir': '/content/data/score_vectors'
    },
    'model': {'hidden_dim': 128, 'n_modules': 5},
    'training': {
        'batch_size': 256, 'epochs': 200, 'lr': 1e-3,
        'weight_decay': 1e-5, 'confidence_penalty': 0.1
    },
    'compute': {'num_workers': 0, 'pin_memory': False, 'persistent_workers': False}
}
with open('/content/SentinelID/configs/fusion_config.yaml', 'w') as f:
    yaml.dump(cfg_fusion, f)
!cd /content/SentinelID && python training/train_fusion.py --config configs/fusion_config.yaml

## M7 · Edge Distillation → ONNX

In [ ]:
cfg_distill = {
    'project': {'name': 'sentinelid-distill', 'device': 'cuda',
                'mixed_precision': True, 'compile_model': False},
    'paths': {'checkpoint_dir': str(DRIVE_CKPT)},
    'face': {'num_classes': 200},
    'student': {'face_embed_dim': 256, 'au_out': 1},
    'data': {
        'image_size': 224,
        'datasets': [{'name': 'liveness_train', 'root': '/content/data/liveness'}]
    },
    'training': {
        'batch_size': 64, 'epochs': 30, 'lr': 1e-3,
        'weight_decay': 1e-4, 'temperature': 4.0,
        'alpha': 0.7, 'embedding_weight': 0.3
    },
    'compute': {'num_workers': 2, 'pin_memory': True, 'persistent_workers': True}
}
with open('/content/SentinelID/configs/distillation_config.yaml', 'w') as f:
    yaml.dump(cfg_distill, f)
print('Distillation config written.')

In [ ]:
!pip install -q onnxscript
!cd /content/SentinelID && python training/distill_edge.py --config configs/distillation_config.yaml

## ✅ Verify All Checkpoints

In [ ]:
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/sentinelid/checkpoints')
expected = ['liveness_best.pt', 'deepfake_best.pt', 'face_model.pt',
            'behavioral_best.pt', 'document_best.pt', 'fusion_best.pt',
            'edge_model.pt', 'edge_model.onnx']

print('─' * 55)
print(f'{"Checkpoint":<30} {"Size":>10} {"Status"}')
print('─' * 55)
all_ok = True
for name in expected:
    p = ckpt_dir / name
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f'{name:<30} {size_mb:>9.1f}M  ✓')
    else:
        print(f'{name:<30} {"MISSING":>10}  ✗')
        all_ok = False
print('─' * 55)
print('All checkpoints present! ✓' if all_ok else 'Some checkpoints missing — re-run the relevant cells.')

## 🔍 ONNX Inference Sanity Check
Verify the exported ONNX model produces sensible outputs.

In [ ]:
import onnxruntime as ort
import numpy as np
from pathlib import Path

onnx_path = str(Path('/content/drive/MyDrive/sentinelid/checkpoints') / 'edge_model.onnx')
sess = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

print('Inputs:')
for inp in sess.get_inputs():
    print(f'  {inp.name}: {inp.shape} ({inp.type})')
print('Outputs:')
for out in sess.get_outputs():
    print(f'  {out.name}: {out.shape} ({out.type})')

# Run with random batch
dummy = np.random.randn(4, 3, 224, 224).astype(np.float32)
outs = sess.run(None, {'image': dummy})
liveness, embed, au, logits_live, logits_au = outs
print(f'\nBatch=4 forward pass:')
print(f'  liveness_score: {liveness.shape} | mean={liveness.mean():.3f}')
print(f'  face_embed:     {embed.shape}    | norm={np.linalg.norm(embed, axis=1).mean():.3f} (should ≈1)')
print(f'  au_signal:      {au.shape}')
print('\n✓ ONNX model verified and ready for mobile deployment.')

## 📊 Push final state to Git

In [ ]:
import subprocess

repo = '/content/SentinelID'

subprocess.run(['git', 'config', 'user.email', 'aprameya.bharadwaj.05@gmail.com'], cwd=repo)
subprocess.run(['git', 'config', 'user.name', 'Aprameya Bharadwaj'], cwd=repo)

result = subprocess.run(
    ['git', 'add', 'configs/', 'models/', 'training/'],
    cwd=repo, capture_output=True, text=True
)
print(result.stdout, result.stderr)

result = subprocess.run(
    ['git', 'commit', '-m', 'fix: all 7 modules trained; ONNX export working'],
    cwd=repo, capture_output=True, text=True
)
print(result.stdout, result.stderr)

result = subprocess.run(['git', 'push'], cwd=repo, capture_output=True, text=True)
print(result.stdout, result.stderr)